# Multi-Conference SAC/AC Matching

This notebook runs two venue-specific SAC/AC matchings against one ARR venue.

Flow:
1. Run the ARR matching setup invitations and wait for them to complete.
2. Block the opposite preferred-venue submissions with `Area_Chairs/-/Custom_User_Demands` edges of weight `0`.
3. Exclude the opposite SAC roster through `exclude_sacs` in the `SACACMatching` checkpoint.
4. Run the full `SACACMatching` pipeline for conference 1.
5. Zero `Area_Chairs/-/Custom_Max_Papers` for ACs already used in conference 1.
6. Swap the blocked submissions and excluded SAC roster, then run conference 2.
7. Clear the temporary AC constraints. Optionally merge the two labels, but the default recommendation is to deploy the split labels separately.

Note: the second round should block conference 1 submissions, not conference 2 again.

In [ ]:
import time
from copy import deepcopy

import openreview

from core import SACACMatching
from edge_utils import delete_edges_with_polling

## Parameters

In [ ]:
VENUE_ID = "aclweb.org/ACL/ARR/2025/October"

BASEURLS = [
    "https://api.openreview.net",
    "https://api2.openreview.net",
]
MATCHER_BASEURL = "https://api2.openreview.net"

CONFERENCE_ONE_NAME = "ACL"
CONFERENCE_TWO_NAME = "EACL"
CONFERENCE_ONE_PREFERRED_VENUES = {CONFERENCE_ONE_NAME}
CONFERENCE_TWO_PREFERRED_VENUES = {CONFERENCE_TWO_NAME}
TREAT_OTHER_PREFERENCES_AS_CONFERENCE_TWO = True ## default other conferences to the second
PREFERRED_VENUE_FIELD = "preferred_venue"

# Fill these with profile ids from the shared venue SAC pool. Could also be loaded from a file
CONFERENCE_ONE_SACS = []
CONFERENCE_TWO_SACS = []

CONFERENCE_ONE_AC_TITLE = "ac-matching-acl"
CONFERENCE_ONE_SAC_TITLE = "sac-matching-acl"
CONFERENCE_TWO_AC_TITLE = "ac-matching-eacl"
CONFERENCE_TWO_SAC_TITLE = "sac-matching-eacl"

CUTOFF = 0
THRESHOLD = 1
TOP_N = 500

SETUP_MATCHING_INVITATIONS = [
    "Setup_SAE_Matching",
    "Setup_AE_Matching",
    "Setup_Reviewer_Matching",
]
SETUP_MATCHING_TIMEOUT_SECONDS = 600
SETUP_MATCHING_POLL_INTERVAL_SECONDS = 5

## deprecated, but explicitly setting false
PRIORITY_TRACK_LOADS = {"enabled": False}

# Optional resume checkpoints for partially completed runs.
CONFERENCE_ONE_CHECKPOINT = {}
CONFERENCE_TWO_CHECKPOINT = {}

# Optional merged labels. Leave as None if you plan to deploy split labels separately.
MERGED_AC_TITLE = None
MERGED_SAC_TITLE = None

In [ ]:
client_v1 = openreview.Client(
    username='',
    password='',
    baseurl='https://api.openreview.net'
)
client = openreview.api.OpenReviewClient(
    username='',
    password='',
    baseurl='https://api2.openreview.net'
)
domain = client.get_group(VENUE_ID)
REQUEST_FORM_ID = domain.content['request_form_id']['value']
venue = openreview.helpers.get_conference(client_v1, REQUEST_FORM_ID)

print(f"Venue: {venue.id}")
print(f"Request form: {REQUEST_FORM_ID}")

In [ ]:
def note_value(note, field, default=None):
    raw = note.content.get(field, default)
    if isinstance(raw, dict):
        return raw.get("value", default)
    return raw


def _latest_log(logs):
    return max(logs, key=lambda log: log.get("sdate") or log.get("cdate") or log.get("tcdate") or 0)


def wait_for_process_completion(edit_id, invitation_id, min_sdate):
    deadline = time.time() + SETUP_MATCHING_TIMEOUT_SECONDS
    last_logs = []
    while time.time() < deadline:
        logs = client.get_process_logs(id=edit_id)
        if not logs:
            logs = client.get_process_logs(invitation=invitation_id, min_sdate=min_sdate)
        last_logs = logs
        finished_logs = [log for log in logs if log.get("status") in {"ok", "error"}]
        if finished_logs:
            latest = _latest_log(finished_logs)
            if latest.get("status") == "ok":
                print(f"{invitation_id} setup completed")
                return latest
            raise openreview.OpenReviewException(latest.get("log", f"{invitation_id} setup failed"))
        time.sleep(SETUP_MATCHING_POLL_INTERVAL_SECONDS)
    raise TimeoutError(
        f"Timed out waiting for {invitation_id}; last logs: {last_logs[:3]}"
    )


def trigger_matching_setup_invitation(invitation_name):
    invitation_id = f"{venue.id}/-/{invitation_name}"
    invitation = client.get_invitation(invitation_id)
    current_count = note_value(invitation, "count", 0)
    min_sdate = int(time.time() * 1000)
    edit = client.post_invitation_edit(
        invitations=venue.get_meta_invitation_id(),
        readers=[venue.id],
        writers=[venue.id],
        signatures=[venue.id],
        invitation=openreview.api.Invitation(
            id=invitation_id,
            content={"count": {"value": current_count + 1}},
        ),
    )
    return wait_for_process_completion(edit["id"], invitation_id, min_sdate)


def run_matching_setup_scripts():
    logs = {}
    for invitation_name in SETUP_MATCHING_INVITATIONS:
        print(f"Running {invitation_name}")
        logs[invitation_name] = trigger_matching_setup_invitation(invitation_name)
    return logs


def partition_submissions():
    conference_one_ids = []
    conference_two_ids = []
    unclassified = []
    submissions = client.get_all_notes(invitation=f"{venue.id}/-/Submission")
    for note in submissions:
        preferred_venue = note_value(note, PREFERRED_VENUE_FIELD)
        if preferred_venue in CONFERENCE_ONE_PREFERRED_VENUES:
            conference_one_ids.append(note.id)
        elif preferred_venue in CONFERENCE_TWO_PREFERRED_VENUES or TREAT_OTHER_PREFERENCES_AS_CONFERENCE_TWO:
            conference_two_ids.append(note.id)
        else:
            unclassified.append((note.id, preferred_venue))
    if unclassified:
        raise ValueError(f"Found submissions outside both buckets: {unclassified[:10]}")
    return submissions, conference_one_ids, conference_two_ids


def clear_ac_custom_user_demands():
    invitation = f"{venue.get_area_chairs_id()}/-/Custom_User_Demands"
    if client.get_edges_count(invitation=invitation):
        delete_edges_with_polling(client, invitation=invitation)


def post_ac_zero_demand_edges(submission_ids):
    invitation = f"{venue.get_area_chairs_id()}/-/Custom_User_Demands"
    readers = [venue.id, venue.get_area_chairs_id()]
    edges = [
        openreview.api.Edge(
            invitation=invitation,
            head=submission_id,
            tail=venue.get_area_chairs_id(),
            weight=0,
            readers=readers,
            writers=[venue.id],
            signatures=[venue.id],
        )
        for submission_id in submission_ids
    ]
    clear_ac_custom_user_demands()
    if edges:
        openreview.tools.post_bulk_edges(client, edges)
    print(f"Posted {len(edges)} zero-demand AC edges")


def name_to_profile_id(members):
    profiles = openreview.tools.get_profiles(client, members)
    mapping = {}
    for profile in profiles:
        for name_obj in profile.content.get("names", []):
            username = name_obj.get("username")
            if username:
                mapping[username] = profile.id
    return mapping


def reset_ac_custom_max_papers_from_load_notes():
    role_id = venue.get_area_chairs_id()
    invitation = f"{role_id}/-/Custom_Max_Papers"
    readers = [venue.id, venue.get_senior_area_chairs_id()]
    username_to_id = name_to_profile_id(client.get_group(role_id).members)
    load_notes = client.get_all_notes(invitation=f"{role_id}/-/Max_Load_And_Unavailability_Request")
    edges = []
    for note in load_notes:
        ac_id = username_to_id.get(note.signatures[0])
        if not ac_id:
            continue
        max_load = int(note.content["maximum_load_this_cycle"]["value"])
        edges.append(
            openreview.api.Edge(
                invitation=invitation,
                head=role_id,
                tail=ac_id,
                weight=max_load,
                readers=readers + [ac_id],
                writers=[venue.id],
                signatures=[venue.id],
            )
        )
    if client.get_edges_count(invitation=invitation):
        delete_edges_with_polling(client, invitation=invitation, soft_delete=True)
    if edges:
        openreview.tools.post_bulk_edges(client, edges)
    print(f"Reset {len(edges)} AC Custom_Max_Papers edges from load notes")


def get_assigned_tails(role_id, label):
    return [
        group["id"]["tail"]
        for group in client.get_grouped_edges(
            invitation=f"{role_id}/-/Proposed_Assignment",
            groupby="tail",
            label=label,
        )
    ]


def zero_ac_custom_max_papers(ac_ids):
    invitation = f"{venue.get_area_chairs_id()}/-/Custom_Max_Papers"
    readers = [venue.id, venue.get_senior_area_chairs_id()]
    for ac_id in ac_ids:
        if client.get_edges_count(invitation=invitation, tail=ac_id):
            delete_edges_with_polling(
                client,
                invitation=invitation,
                tail=ac_id,
                soft_delete=True,
                poll_interval_seconds=5,
                max_polls=12,
            )
    edges = [
        openreview.api.Edge(
            invitation=invitation,
            head=venue.get_area_chairs_id(),
            tail=ac_id,
            weight=0,
            readers=readers + [ac_id],
            writers=[venue.id],
            signatures=[venue.id],
        )
        for ac_id in ac_ids
    ]
    if edges:
        openreview.tools.post_bulk_edges(client, edges)
    print(f"Zeroed AC Custom_Max_Papers for {len(edges)} assigned ACs")


def build_checkpoint(excluded_sacs, extra_checkpoint=None):
    checkpoint = {
        "exclude_sacs": list(excluded_sacs),
        "top_n": TOP_N,
        "priority_track_loads": deepcopy(PRIORITY_TRACK_LOADS),
        "reset_ac_tracks": False,
        "reset_sac_tracks": False,
    }
    if extra_checkpoint:
        checkpoint.update(deepcopy(extra_checkpoint))
    checkpoint["exclude_sacs"] = list(excluded_sacs)
    checkpoint.setdefault("top_n", TOP_N)
    checkpoint.setdefault("priority_track_loads", deepcopy(PRIORITY_TRACK_LOADS))
    return checkpoint


def run_split_round(blocked_submission_ids, excluded_sacs, ac_title, sac_title, checkpoint_overrides=None):
    post_ac_zero_demand_edges(blocked_submission_ids)
    checkpoint = build_checkpoint(excluded_sacs, checkpoint_overrides)
    matching = SACACMatching(
        client_v1,
        client,
        REQUEST_FORM_ID,
        matcher_baseurl=MATCHER_BASEURL,
        cutoff=CUTOFF,
        checkpoint=checkpoint,
    )
    matching.run_matching(
        threshold=THRESHOLD,
        ac_title=ac_title,
        sac_title=sac_title,
    )
    return matching


def matching_summary(role_id, label):
    grouped = list(
        client.get_grouped_edges(
            invitation=f"{role_id}/-/Proposed_Assignment",
            groupby="tail",
            label=label,
            select="head",
        )
    )
    return {
        "label": label,
        "assignees": len(grouped),
        "assignments": sum(len(group["values"]) for group in grouped),
    }


def print_summary():
    for role_id, label in [
        (venue.get_area_chairs_id(), CONFERENCE_ONE_AC_TITLE),
        (venue.get_area_chairs_id(), CONFERENCE_TWO_AC_TITLE),
        (venue.get_senior_area_chairs_id(), CONFERENCE_ONE_SAC_TITLE),
        (venue.get_senior_area_chairs_id(), CONFERENCE_TWO_SAC_TITLE),
    ]:
        print(matching_summary(role_id, label))

In [ ]:
matching_setup_logs = run_matching_setup_scripts()

all_sacs = set(client.get_group(venue.get_senior_area_chairs_id()).members)
conference_one_sacs = set(CONFERENCE_ONE_SACS)
conference_two_sacs = set(CONFERENCE_TWO_SACS)

assert conference_one_sacs.isdisjoint(conference_two_sacs), "Conference SAC rosters must be disjoint"
assert conference_one_sacs.issubset(all_sacs), "Conference 1 SACs must be members of the venue SAC group"
assert conference_two_sacs.issubset(all_sacs), "Conference 2 SACs must be members of the venue SAC group"

submissions, conference_one_submission_ids, conference_two_submission_ids = partition_submissions()
print(f"{CONFERENCE_ONE_NAME}-preferred submissions: {len(conference_one_submission_ids)}")
print(f"{CONFERENCE_TWO_NAME}/other submissions: {len(conference_two_submission_ids)}")
print(f"Total submissions: {len(submissions)}")
print(f"{CONFERENCE_ONE_NAME} SACs: {len(conference_one_sacs)}")
print(f"{CONFERENCE_TWO_NAME} SACs: {len(conference_two_sacs)}")

## Conference 1 Round

In [ ]:
reset_ac_custom_max_papers_from_load_notes()

conference_one_matching = run_split_round(
    blocked_submission_ids=conference_two_submission_ids,
    excluded_sacs=conference_two_sacs,
    ac_title=CONFERENCE_ONE_AC_TITLE,
    sac_title=CONFERENCE_ONE_SAC_TITLE,
    checkpoint_overrides=CONFERENCE_ONE_CHECKPOINT,
)

## Freeze ACs Used In Conference 1

In [ ]:
conference_one_ac_ids = get_assigned_tails(venue.get_area_chairs_id(), CONFERENCE_ONE_AC_TITLE)
print(f"{CONFERENCE_ONE_NAME} round assigned {len(conference_one_ac_ids)} ACs")
zero_ac_custom_max_papers(conference_one_ac_ids)

## Conference 2 Round

In [ ]:
conference_two_matching = run_split_round(
    blocked_submission_ids=conference_one_submission_ids,
    excluded_sacs=conference_one_sacs,
    ac_title=CONFERENCE_TWO_AC_TITLE,
    sac_title=CONFERENCE_TWO_SAC_TITLE,
    checkpoint_overrides=CONFERENCE_TWO_CHECKPOINT,
)

## Summary And Cleanup

In [ ]:
print_summary()

# These were temporary constraints for the split runs only.
clear_ac_custom_user_demands()
reset_ac_custom_max_papers_from_load_notes()

## Optional Merge Helper

The default recommendation is to deploy the split labels separately. If you still want a single AC label and a single SAC label, set `MERGED_AC_TITLE` and `MERGED_SAC_TITLE` above and run the next cells.

In [ ]:
def merge_proposed_matchings(role_id, label_one, label_two, merged_label, top_n=200, dry_run=True):
    edge_readers = [venue.id]
    if role_id == venue.get_area_chairs_id():
        edge_readers.append(venue.get_senior_area_chairs_id())

    affinity_scores = {
        group["id"]["tail"]: group["values"]
        for group in client.get_grouped_edges(
            invitation=f"{role_id}/-/Affinity_Score",
            groupby="tail",
            select="head,weight",
        )
    }
    conflicts_by_head = {
        group["id"]["head"]: [edge["tail"] for edge in group["values"]]
        for group in client.get_grouped_edges(
            invitation=f"{role_id}/-/Conflict",
            groupby="head",
            select="tail",
        )
    }
    research_area_heads = {
        group["id"]["tail"]: [edge["head"] for edge in group["values"]]
        for group in client.get_grouped_edges(
            invitation=f"{role_id}/-/Research_Area",
            groupby="tail",
            select="head",
        )
    }
    submission_numbers = {
        submission.id: submission.number
        for submission in client.get_all_notes(invitation=f"{venue.id}/-/Submission")
    }

    proposed_assignments = {}
    for label in [label_one, label_two]:
        for group in client.get_grouped_edges(
            invitation=f"{role_id}/-/Proposed_Assignment",
            groupby="tail",
            label=label,
            select="head",
        ):
            proposed_assignments.setdefault(group["id"]["tail"], [])
            proposed_assignments[group["id"]["tail"]].extend(edge["head"] for edge in group["values"])

    for assignee, paper_ids in list(proposed_assignments.items()):
        deduped = []
        seen = set()
        for paper_id in paper_ids:
            if paper_id in seen:
                continue
            if paper_id in conflicts_by_head and assignee in conflicts_by_head[paper_id]:
                continue
            seen.add(paper_id)
            deduped.append(paper_id)
        proposed_assignments[assignee] = deduped

    affinity_lookup = {
        assignee: {edge["head"]: edge["weight"] for edge in edges}
        for assignee, edges in affinity_scores.items()
    }

    aggregate_edges = []
    assignment_edges = []
    for assignee, edges in affinity_scores.items():
        sorted_edges = sorted(edges, key=lambda edge: edge["weight"], reverse=True)
        track_heads = set(research_area_heads.get(assignee, []))
        for edge in sorted_edges[:top_n]:
            score = edge["weight"] + (1 if edge["head"] in track_heads else 0)
            aggregate_edges.append(
                openreview.api.Edge(
                    invitation=f"{role_id}/-/Aggregate_Score",
                    head=edge["head"],
                    tail=assignee,
                    weight=score,
                    label=merged_label,
                    readers=edge_readers + [assignee],
                    writers=[venue.id],
                    signatures=[venue.id],
                )
            )

    for assignee, paper_ids in proposed_assignments.items():
        track_heads = set(research_area_heads.get(assignee, []))
        for paper_id in paper_ids:
            submission_number = submission_numbers.get(paper_id)
            if submission_number is None:
                continue
            score = affinity_lookup.get(assignee, {}).get(paper_id, 0)
            if paper_id in track_heads:
                score += 1
            assignment_edges.append(
                openreview.api.Edge(
                    invitation=f"{role_id}/-/Proposed_Assignment",
                    head=paper_id,
                    tail=assignee,
                    weight=score,
                    label=merged_label,
                    readers=edge_readers + [assignee],
                    nonreaders=[f"{venue.id}/Submission{submission_number}/Authors"],
                    writers=[venue.id],
                    signatures=[venue.id],
                )
            )

    print(
        f"Prepared {len(aggregate_edges)} aggregate scores and {len(assignment_edges)} assignments for {merged_label}"
    )
    if dry_run:
        return aggregate_edges, assignment_edges

    if client.get_edges_count(invitation=f"{role_id}/-/Aggregate_Score", label=merged_label):
        delete_edges_with_polling(
            client,
            invitation=f"{role_id}/-/Aggregate_Score",
            label=merged_label,
        )
    openreview.tools.post_bulk_edges(client, aggregate_edges)

    if client.get_edges_count(invitation=f"{role_id}/-/Proposed_Assignment", label=merged_label):
        delete_edges_with_polling(
            client,
            invitation=f"{role_id}/-/Proposed_Assignment",
            label=merged_label,
        )
    openreview.tools.post_bulk_edges(client, assignment_edges)

    return aggregate_edges, assignment_edges

In [ ]:
if MERGED_SAC_TITLE:
    merge_proposed_matchings(
        venue.get_senior_area_chairs_id(),
        CONFERENCE_ONE_SAC_TITLE,
        CONFERENCE_TWO_SAC_TITLE,
        MERGED_SAC_TITLE,
        top_n=TOP_N,
        dry_run=False,
    )

if MERGED_AC_TITLE:
    merge_proposed_matchings(
        venue.get_area_chairs_id(),
        CONFERENCE_ONE_AC_TITLE,
        CONFERENCE_TWO_AC_TITLE,
        MERGED_AC_TITLE,
        top_n=TOP_N,
        dry_run=False,
    )

## Deployment

Recommended: validate the split labels on staging, then deploy the split AC labels and split SAC labels separately. If you created merged labels instead, deploy the merged labels.

In [ ]:
# Recommended split deployment path.
venue.set_assignments(CONFERENCE_ONE_AC_TITLE, venue.get_area_chairs_id(), overwrite=False)
venue.set_assignments(CONFERENCE_TWO_AC_TITLE, venue.get_area_chairs_id(), overwrite=False)
venue.set_assignments(CONFERENCE_ONE_SAC_TITLE, venue.get_senior_area_chairs_id(), overwrite=False)
venue.set_assignments(CONFERENCE_TWO_SAC_TITLE, venue.get_senior_area_chairs_id(), overwrite=False)

# If you merged instead:
if MERGED_AC_TITLE:
    venue.set_assignments(MERGED_AC_TITLE, venue.get_area_chairs_id(), overwrite=True)
if MERGED_SAC_TITLE:
    venue.set_assignments(MERGED_SAC_TITLE, venue.get_senior_area_chairs_id(), overwrite=True)